# Individual Tree Crown Delineation (ITCD) Inference Pipeline
This notebook executes the full end-to-end inference workflow: **Raster Tiling -> Detectron2 Inference -> Polygon Conversion -> Georeferencing -> Stitching & Spatial NMS -> GeoPackage (.gpkg) Export**.

### 1. Environment Setup & Dependency Installation

In [ ]:
import warnings
warnings.simplefilter('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install geospatial libraries
!pip install segment-geospatial pycocotools geopandas rasterio

# Install Detectron2
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup('./detectron2/setup.py')
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

### 2. Clone and Install the ITCD Package

In [ ]:
# Clone the repository and install:
!git clone https://github.com/ambarwariagus/itcd-domain-aware.git
%cd itcd-domain-aware
!pip install -e .

import sys, os
if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

### 3. Pipeline Configuration & Paths

In [ ]:
from itcd import ITCDInferenceConfig, run_full_pipeline

# Configure model weights, thresholds, and spatial parameters
config = ITCDInferenceConfig(
    trained_model_dir='/content/drive/MyDrive/ITCD_Domain_Aware/models/250525_R50_RESTOR_fold_0',
    score_thresh_test=0.5,
    nms_thresh_test=0.3,
    tile_size=1024,
    overlap=256,
    edge_shift_meters=1.0,
    spatial_iou_threshold=0.5,
    spatial_conf_threshold=0.5,
    area_threshold_m2=2.0
)

# Define input orthophoto and output destinations
IMAGE_INPUT = '/content/drive/MyDrive/ITCD_Domain_Aware/dataset/sample/cmh01.tif'
WORK_DIR    = '/content/drive/MyDrive/ITCD_Domain_Aware/dataset/sample/tiles'
OUTPUT_GPKG = '/content/drive/MyDrive/ITCD_Domain_Aware/dataset/sample/tiles/tree_crowns.gpkg'

### 4. Execute the End-to-End Pipeline

In [ ]:
# Execute the complete pipeline in a single automated step
cleaned_gdf = run_full_pipeline(
    orthophoto_path=IMAGE_INPUT,
    work_dir=WORK_DIR,
    config=config,
    output_gpkg=OUTPUT_GPKG
)

print("Preview detected tree crowns:")
display(cleaned_gdf.head())

### 5. (Optional) Step-by-Step Modular Execution

In [ ]:
# To run individual pipeline stages manually:
from itcd import (
    run_tiling,
    ITCDDetector,
    convert_rle_to_labelme,
    batch_labelme_to_geojson,
    stitch_and_clean_pipeline
)

# Stage 1: Tiling (Split raster)
# run_tiling(IMAGE_INPUT, WORK_DIR, tile_size=config.tile_size, overlap=config.overlap)

# Stage 2: Detectron2 Inference
# detector = ITCDDetector(config.config_file, config.model_weights, config.score_thresh_test, config.nms_thresh_test)
# detector.predict_directory(WORK_DIR, f"{WORK_DIR}/predictions")

# Stage 3: Convert RLE Masks to LabelMe JSON
# convert_rle_to_labelme(f"{WORK_DIR}/predictions", f"{WORK_DIR}/labelme_output")

# Stage 4: Georeferencing to GeoJSON
# batch_labelme_to_geojson(f"{WORK_DIR}/labelme_output", WORK_DIR, f"{WORK_DIR}/geojson")

# Stage 5: Spatial NMS & Export to GeoPackage
# stitch_and_clean_pipeline(f"{WORK_DIR}/geojson", WORK_DIR, OUTPUT_GPKG, shift=1.0)